# InternNova — Data Analytics Internship
## Week 6 Assignment: Dataset Selection & Data Preparation 
**Dataset:** `sales_fact.csv` & `product.csv` \
**Prepared by:** Brojo Mohan Dutta\
**Environment:** Anaconda / Jupyter Notebook 7.4.5 / Python 3.x / Pandas

# Setup & Load Data

#### Explanation:
Imports required libraries and loads both raw CSVs.

In [20]:
import pandas as pd
import numpy as np

sales = pd.read_csv("sales_fact.csv")
products = pd.read_csv("products.csv")

sales.head()

,sale_id,order_date,product_id,region,payment_method,quantity,unit_price,sales_amount,notes
0,5001,2024-06-01,P05,West,Debit Card,9.0,3299.0,29691.0,NaN
1,5002,2024-06-03,P10,North,Debit Card,4.0,14999.0,59996.0,NaN
2,5003,2024-06-05,P07,West,UPI,8.0,6999.0,55992.0,NaN
3,5004,2024-06-07,P04,South,Debit Card,8.0,1999.0,15992.0,NaN
4,5005,2024-06-09,P04,West,UPI,5.0,NaN,NaN,NaN


# Data Inspection

#### Explanation: 
Initial inspection of the raw sales dataset by checking its dimensions, column names, data types, missing values, duplicate records, and unique region values. The results help identify data-quality issues that need to be addressed during the cleaning process.

In [21]:
print("Shape:", sales.shape)
print("Columns:", sales.columns.tolist())
print("Data Types:\n", sales.dtypes)
print("Missing values:\n", sales.isnull().sum())
print("Duplicate rows:", sales.duplicated().sum())
print("Unique region values:", sales["region"].unique())

Shape: (61, 9)
Columns: ['sale_id', 'order_date', 'product_id', 'region', 'payment_method', 'quantity', 'unit_price', 'sales_amount', 'notes']
Data Types:
 sale_id             int64
order_date         object
product_id         object
region             object
payment_method     object
quantity          float64
unit_price        float64
sales_amount      float64
notes             float64
dtype: object
Missing values:
 sale_id            0
order_date         0
product_id         0
region             0
payment_method     0
quantity           1
unit_price         1
sales_amount       2
notes             61
dtype: int64
Duplicate rows: 1
Unique region values: ['West' 'North' 'South' 'East' 'north' 'SOUTH']


# Data Cleaning

#### Explanation: 
Replicates every Power Query transformation in pandas — drop the unnecessary column, fix data types, standardize text, fill nulls, correct the negative-quantity error, and recompute sales_amount.

In [22]:
df = sales.copy()

# 1. Remove unnecessary column
df = df.drop(columns=["notes"])

# 2. Fix data types
df["order_date"] = pd.to_datetime(df["order_date"])

# 3. Standardize inconsistent text in 'region'
df["region"] = df["region"].str.strip().str.title()

# 4. Handle missing values
df["quantity"] = df["quantity"].fillna(5)
df["unit_price"] = df["unit_price"].fillna(1999)

# 5. Fix incorrect value (negative quantity -> positive)
df.loc[df["quantity"] < 0, "quantity"] = df.loc[df["quantity"] < 0, "quantity"].abs()
df["quantity"] = df["quantity"].astype(int)

# 6. Recompute sales_amount cleanly
df["sales_amount"] = df["quantity"] * df["unit_price"]

# 7. Remove duplicate records
df = df.drop_duplicates()

print("Shape after cleaning:", df.shape)
print("Missing values after cleaning:\n", df.isnull().sum())
print("Duplicate rows after cleaning:", df.duplicated().sum())

Shape after cleaning: (60, 8)
Missing values after cleaning:
 sale_id           0
order_date        0
product_id        0
region            0
payment_method    0
quantity          0
unit_price        0
sales_amount      0
dtype: int64
Duplicate rows after cleaning: 0


# Merge with Products & Calculate Profit

#### Explanation: 
Combining the cleaned sales data with the product information using product_id as the common key. It then calculates total_cost from quantity and unit cost, and calculates transaction-level profit by subtracting total cost from sales amount. Finally, the resulting 60-row, 13-column dataset is exported as sales_data_cleaned.csv for further EDA and analysis.

In [23]:
df = df.merge(products, on="product_id", how="left")

df["total_cost"] = df["quantity"] * df["unit_cost"]
df["profit"] = df["sales_amount"] - df["total_cost"]

# Export the cleaned and merged dataset for EDA
eda_file = "sales_data_cleaned.csv"

df.to_csv(eda_file, index=False)

print(f"EDA dataset exported successfully: {eda_file}")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

EDA dataset exported successfully: sales_data_cleaned.csv
Rows: 60
Columns: 13


# Verify Cleaned Dataset

#### Explanation: 
Verifies that the exported cleaned dataset has the expected structure by checking its number of rows and columns, column names, sample records, and missing values. This confirms that the dataset is ready for further Exploratory Data Analysis (EDA).

In [28]:
# Verify the exported EDA dataset
eda_df = pd.read_csv("sales_data_cleaned.csv")

print("EDA Dataset Shape:", eda_df.shape)
print("\nColumns:")
print(eda_df.columns.tolist())

print("\nFirst 5 Rows:")
display(eda_df.head())

print("\nMissing Values:")
print(eda_df.isnull().sum())

EDA Dataset Shape: (60, 13)

Columns:
['sale_id', 'order_date', 'product_id', 'region', 'payment_method', 'quantity', 'unit_price', 'sales_amount', 'product_name', 'category', 'unit_cost', 'total_cost', 'profit']

First 5 Rows:


,sale_id,order_date,product_id,region,payment_method,quantity,unit_price,sales_amount,product_name,category,unit_cost,total_cost,profit
0,5001,2024-06-01,P05,West,Debit Card,9,3299.0,29691.0,Webcam HD,Electronics,2200,19800,9891.0
1,5002,2024-06-03,P10,North,Debit Card,4,14999.0,59996.0,Gaming Chair,Furniture,8900,35600,24396.0
2,5003,2024-06-05,P07,West,UPI,8,6999.0,55992.0,Noise Cancelling Headphones,Electronics,4200,33600,22392.0
3,5004,2024-06-07,P04,South,Debit Card,8,1999.0,15992.0,Laptop Stand,Accessories,1100,8800,7192.0
4,5005,2024-06-09,P04,West,UPI,5,1999.0,9995.0,Laptop Stand,Accessories,1100,5500,4495.0



Missing Values:
sale_id           0
order_date        0
product_id        0
region            0
payment_method    0
quantity          0
unit_price        0
sales_amount      0
product_name      0
category          0
unit_cost         0
total_cost        0
profit            0
dtype: int64
